# Spatial CCC true-neighbor vs far-pseudo-edge benchmark for SpiderNet

This notebook implements a focused spatial false-positive / spatial-null benchmark for **HGSOC** and **aging mouse brain** SpiderNet studies.

## Experiment — true spatial neighbor edges vs far pseudo-edges

For each study, the notebook identifies, for each MI dimension, the dominant sender→receiver cell-type pair(s) with the strongest original mean MI activity on true spatial neighbor edges. It then compares SpiderNet-inferred raw MI activity between:

1. **True neighbor edges** used by SpiderNet.
2. **Far pseudo-edges** with the same sender cell and same receiver cell type, where the far receiver is outside the sender's KNN-1000 spatial neighborhood.

The key output is:

```text
log2FC = log2((mean MI activity on true neighbor edges + eps)
              / (mean MI activity on far pseudo-edges + eps))
```

The final figure is a single PDF with the AgingMousebrain and HGSOC horizontal log2FC barplots arranged as two rows, one study per row.


## 0. Paths and analysis controls

Paths come from `benchmark_config.py`. Run this notebook with the benchmark folder as the working directory. The default mode recomputes the spatial control from trained checkpoints. Set `SPIDERNET_PLOT_ONLY=1` before starting the notebook to reuse the per-study summary CSVs and regenerate the combined figure.


In [ ]:
import os

from benchmark_config import BENCHMARK_DIR, DATA_ROOT, RESULTS_ROOT, add_spidernet_to_path

# ============================================================
# User controls
# ============================================================

STUDIES_TO_RUN = ["AgingMousebrain", "HGSOC"]  # both studies run sequentially

# Far pseudo-edge definition
FAR_K = 1000  # far pseudo-edge receiver must be outside sender's KNN-1000 neighborhood

# For each MI, compare only the top N sender→receiver cell-type pairs
# ranked by original mean MI activity on true spatial neighbor edges.
TOP_N_PAIRS_PER_MI = 1

# Statistic used to define dominant MI-pair axes and compute near/far log2FC.
# Options:
#   "mean"   -> log2FC of mean MI activity
#   "median" -> log2FC of median MI activity
MI_ACTIVITY_COMPARISON_STAT = "mean"


# Sampling controls for true-neighbor edges in each selected sender→receiver pair.
# Set to None to use all edges for each selected pair.
MAX_TRUE_EDGES_PER_CELLTYPE_PAIR = 50000
MIN_TRUE_EDGES_PER_CELLTYPE_PAIR = 50

# Plot controls
MAX_LABEL_CHARS = 34
COMBINED_BARPLOT_WIDTH = 7.2  # shared width for the two vertically stacked study panels

# Reproducibility
RANDOM_SEED = 123

# Numerical stability for log2 ratio
MI_ACTIVITY_EPS = 1e-6

# Output root
COMBINED_OUTPUT_DIR = BENCHMARK_DIR / "output" / "executed"

# Reuse saved summaries without loading checkpoints or running model inference.
PLOT_ONLY = os.environ.get("SPIDERNET_PLOT_ONLY", "0") == "1"


## 1. Imports and plotting setup
The model uses CUDA when available and otherwise CPU, matching the original inference behavior.


In [ ]:
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
from sklearn.neighbors import NearestNeighbors

import torch

add_spidernet_to_path()

from SpiderNet.config import TrainingConfig
from SpiderNet.io import load_processed_data
from SpiderNet.api import build_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


def set_plot_style():
    plt.close("all")
    plt.style.use("default")
    rcParams.update({
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.family": "Arial",
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "figure.dpi": 150,
        "savefig.dpi": 300,
    })


## 2. Study-specific configuration

In [ ]:
# ============================================================
# Study configs adapted from the HGSOC, AgingBrain, and CCC benchmark notebooks
# ============================================================

STUDY_CONFIG = {
    "HGSOC": {
        "display_name": "HGSC / HGSOC",
        "data_root": DATA_ROOT / "HGSOC",
        "processed_data_dir": RESULTS_ROOT / "HGSOC" / "ProcessedData",
        "output_root": RESULTS_ROOT / "HGSOC",
        "celltype_col": "cell.types",
        "sample_col": "samples",
        "spatial_key": "spatial",
        "dim_envir": 15,
        "hidden_channels": None,
        "preferred_checkpoint": "model_epoch19999.pth",
    },
    "AgingMousebrain": {
        "display_name": "aging mouse brain",
        "data_root": DATA_ROOT / "AgingBrain",
        "processed_data_dir": RESULTS_ROOT / "AgingBrain" / "ProcessedData",
        "output_root": RESULTS_ROOT / "AgingBrain",
        "celltype_col": "celltype",
        "sample_col": "age",
        "spatial_key": "spatial",
        "dim_envir": 30,
        "hidden_channels": 256,
        "preferred_checkpoint": None,
    },
}

for study in STUDIES_TO_RUN:
    if study not in STUDY_CONFIG:
        raise ValueError(f"Unsupported study: {study}. Options: {list(STUDY_CONFIG)}")

STUDY_CONFIG


## 3. Load trained SpiderNet model and processed objects

In [ ]:
def _torch_load_state_dict(path, map_location):
    # Compatible with both older and newer PyTorch.
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def find_latest_checkpoint(model_dir, preferred_name=None):
    model_dir = Path(model_dir)
    if preferred_name is not None:
        preferred = model_dir / preferred_name
        if preferred.exists():
            return preferred

    candidates = sorted(
        model_dir.glob("model_epoch*.pth"),
        key=lambda p: int(re.search(r"model_epoch(\d+)\.pth$", p.name).group(1))
        if re.search(r"model_epoch(\d+)\.pth$", p.name) else -1,
    )
    if len(candidates) == 0:
        raise FileNotFoundError(f"No model_epoch*.pth checkpoint found in {model_dir}")
    return candidates[-1]


def load_run_dirs(output_root):
    output_root = Path(output_root)
    run_dirs_path = output_root / "run_dirs.json"
    if not run_dirs_path.exists():
        raise FileNotFoundError(
            f"Cannot find run_dirs.json: {run_dirs_path}\n"
            "Please run the corresponding training notebook first, or edit STUDY_CONFIG."
        )
    with open(run_dirs_path, "r", encoding="utf-8") as f:
        run_dirs = json.load(f)
    return {k: Path(v) for k, v in run_dirs.items()}


def load_study_resources(study_key, cfg):
    print("\n" + "=" * 80)
    print(f"Loading study: {study_key} ({cfg['display_name']})")
    print("=" * 80)

    run_dirs = load_run_dirs(cfg["output_root"])
    run_dir = Path(run_dirs["run_dir"])
    model_dir = Path(run_dirs.get("model_dir", run_dir / "Model"))

    processed = load_processed_data(cfg["processed_data_dir"])
    adata_list = processed.adata_list
    spidernet_data_list = processed.spidernet_data

    model_config_path = model_dir / "SpiderNet_model_config.json"
    if not model_config_path.exists():
        raise FileNotFoundError(f"Cannot find model config: {model_config_path}")

    with open(model_config_path, "r", encoding="utf-8") as f:
        model_config_dict = json.load(f)
    train_cfg = TrainingConfig(**model_config_dict)

    build_kwargs = {}
    if cfg.get("hidden_channels", None) is not None:
        build_kwargs["hidden_channels"] = cfg["hidden_channels"]

    model = build_model(
        processed=processed,
        train_cfg=train_cfg,
        device=device,
        **build_kwargs,
    )

    ckpt_path = find_latest_checkpoint(
        model_dir=model_dir,
        preferred_name=cfg.get("preferred_checkpoint", None),
    )
    state = _torch_load_state_dict(ckpt_path, map_location=device)
    model.load_state_dict(state)
    model = model.to(device)
    model.eval()

    outdir = run_dir / "Spatial_false_positive_null_benchmark"
    outdir.mkdir(parents=True, exist_ok=True)

    print("Processed data:", cfg["processed_data_dir"])
    print("Run directory:", run_dir)
    print("Model directory:", model_dir)
    print("Checkpoint:", ckpt_path.name)
    print("Output directory:", outdir)
    print("n_slices:", len(adata_list))
    print("n_cells:", int(np.sum([a.n_obs for a in adata_list])))

    return {
        "study_key": study_key,
        "cfg": cfg,
        "run_dirs": run_dirs,
        "run_dir": run_dir,
        "outdir": outdir,
        "processed": processed,
        "adata_list": adata_list,
        "spidernet_data_list": spidernet_data_list,
        "model": model,
        "train_cfg": train_cfg,
    }


## 4. Core utility functions

In [ ]:
def get_edge_index_np(data):
    edge = data["edge_index"] if "edge_index" in data else data.edge_index
    if torch.is_tensor(edge):
        edge_np = edge.detach().cpu().numpy()
    else:
        edge_np = np.asarray(edge)

    if edge_np.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_np.shape}")

    # SpiderNet notebooks use shape (E, 2). Keep support for PyG's conventional (2, E).
    if edge_np.shape[0] == 2 and edge_np.shape[1] != 2:
        edge_np = edge_np.T

    if edge_np.shape[1] != 2:
        raise ValueError(f"edge_index must be shape (E, 2) or (2, E), got {edge_np.shape}")

    return edge_np.astype(np.int64, copy=False)


def format_edge_index_like(edge_index_np, template_data, device=None):
    old_edge = template_data["edge_index"] if "edge_index" in template_data else template_data.edge_index
    if torch.is_tensor(old_edge):
        old_shape = tuple(old_edge.shape)
    else:
        old_shape = tuple(np.asarray(old_edge).shape)

    if len(old_shape) == 2 and old_shape[0] == 2 and old_shape[1] != 2:
        edge_use = edge_index_np.T
    else:
        edge_use = edge_index_np

    return torch.as_tensor(edge_use, dtype=torch.long, device=device)


def get_spatial_matrix(adata, spatial_key="spatial"):
    if spatial_key in adata.obsm:
        coords = np.asarray(adata.obsm[spatial_key])
    else:
        candidates = [
            k for k in adata.obsm_keys()
            if np.asarray(adata.obsm[k]).ndim == 2 and np.asarray(adata.obsm[k]).shape[1] >= 2
        ]
        if len(candidates) == 0:
            raise KeyError(
                f"Cannot find adata.obsm['{spatial_key}'] or any 2D coordinate matrix in adata.obsm."
            )
        coords = np.asarray(adata.obsm[candidates[0]])
        print(f"Using adata.obsm['{candidates[0]}'] as spatial coordinates.")

    if coords.shape[1] > 2:
        coords = coords[:, :2]
    return coords.astype(float, copy=False)


def get_celltype_array(adata, celltype_col):
    if celltype_col not in adata.obs.columns:
        raise KeyError(f"Cannot find cell-type column '{celltype_col}' in adata.obs.")
    return adata.obs[celltype_col].astype(str).to_numpy()


@torch.no_grad()
def run_model_get_mi(model, data, device):
    data_dev = data.to(device)
    model.eval()
    out = model(data_dev)

    if isinstance(out, dict):
        for key in ["factor_envir", "Factor_envir", "mi_activity", "MI_activity"]:
            if key in out:
                factor = out[key]
                break
        else:
            raise KeyError(f"Cannot find MI activity in model output dict keys: {list(out.keys())}")
    elif isinstance(out, (tuple, list)):
        if len(out) >= 5:
            # SpiderNet notebooks use: exp_recon, _, _, _, Factor_envir, _, _, _ = model(data)
            factor = out[4]
        else:
            raise ValueError(f"Model output tuple/list is too short: length={len(out)}")
    else:
        raise TypeError(f"Unsupported model output type: {type(out)}")

    if torch.is_tensor(factor):
        factor = factor.detach().cpu().numpy()
    else:
        factor = np.asarray(factor)

    return factor.astype(np.float32, copy=False)


def infer_original_mi_per_slice(resources):
    """Infer raw MI activities on the original spatial graph for each slice.

    No per-MI normalization is applied here. The raw model outputs are directly
    compared with far-edge or shuffled-expression inference outputs.
    """
    model = resources["model"]
    data_list = resources["spidernet_data_list"]

    original_factors = []
    for slice_index, data in enumerate(data_list):
        print(f"Original inference | {resources['study_key']} slice {slice_index + 1}/{len(data_list)}")
        factor = run_model_get_mi(model, data, device)
        original_factors.append(factor)

    return original_factors


def clone_graph_with_new_edges(data, new_edge_index_np, zero_old_edge_attrs=True):
    """Clone a SpiderNet PyG graph and replace edge_index.

    The SpiderNet forward pass used in the original perturbation notebooks infers MI activity
    from cell expression and edge_index. Edge-level target attributes such as cellpair_LRpair_neigh
    are not needed for MI inference, but if their first dimension equals the old number of edges,
    this helper resizes them to zero arrays to avoid shape mismatch after replacing edge_index.
    """
    data_cpu = data.to("cpu")
    old_edge_np = get_edge_index_np(data_cpu)
    old_E = old_edge_np.shape[0]
    new_E = int(new_edge_index_np.shape[0])

    g = data_cpu.clone()
    g["edge_index"] = format_edge_index_like(new_edge_index_np, template_data=data_cpu, device="cpu")

    if zero_old_edge_attrs:
        for key in list(g.keys()):
            if key == "edge_index":
                continue
            val = g[key]
            if torch.is_tensor(val) and val.ndim >= 1 and val.shape[0] == old_E:
                new_shape = (new_E,) + tuple(val.shape[1:])
                g[key] = torch.zeros(new_shape, dtype=val.dtype, device="cpu")
            elif isinstance(val, np.ndarray) and val.ndim >= 1 and val.shape[0] == old_E:
                new_shape = (new_E,) + tuple(val.shape[1:])
                g[key] = np.zeros(new_shape, dtype=val.dtype)

    return g






def mi_column_names(n_mi):
    return [f"MI-{i + 1}" for i in range(n_mi)]


def safe_log2_ratio(numer, denom, eps=MI_ACTIVITY_EPS):
    return np.log2((np.asarray(numer, dtype=float) + eps) / (np.asarray(denom, dtype=float) + eps))


## 5. Compare true spatial neighbors with matched distant pseudo-edges

Rank sender-to-receiver cell-type pairs by original MI activity, select the dominant context for each MI, and match each retained local edge to a distant edge with the same sender and receiver cell type. The trained model stays fixed. Save per-study ranking, selected contexts, slice-level results, and pooled summary tables before plotting.


In [ ]:
def _mi_num(mi_name):
    try:
        return int(str(mi_name).replace("MI-", ""))
    except Exception:
        return 10**9


def _pair_label_from_edges(edge_index_np, celltypes):
    sender_ct = celltypes[edge_index_np[:, 0]]
    receiver_ct = celltypes[edge_index_np[:, 1]]
    return np.asarray([f"{s}→{r}" for s, r in zip(sender_ct, receiver_ct)], dtype=object)


def _shorten_label(label, max_chars=MAX_LABEL_CHARS):
    label = str(label)
    if len(label) <= max_chars:
        return label
    # Prefer keeping the sender/receiver arrow visible.
    if "→" in label:
        s, r = label.split("→", 1)
        s = s if len(s) <= max_chars // 2 else s[: max_chars // 2 - 1] + "…"
        r = r if len(r) <= max_chars // 2 else r[: max_chars // 2 - 1] + "…"
        return f"{s}→{r}"
    return label[: max_chars - 1] + "…"


def _get_mi_activity_stat():
    stat = str(MI_ACTIVITY_COMPARISON_STAT).strip().lower()
    if stat not in {"mean", "median"}:
        raise ValueError(
            "MI_ACTIVITY_COMPARISON_STAT must be either 'mean' or 'median'. "
            f"Got: {MI_ACTIVITY_COMPARISON_STAT!r}"
        )
    return stat


def _activity_stat_label():
    return _get_mi_activity_stat().capitalize()


def summarize_mi_activity(values, axis=None):
    """Summarize MI activity using the user-specified statistic."""
    stat = _get_mi_activity_stat()
    if stat == "mean":
        return np.nanmean(values, axis=axis)
    if stat == "median":
        return np.nanmedian(values, axis=axis)
    raise ValueError(stat)


def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]
    if values.size == 0:
        return np.nan
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cutoff = 0.5 * np.sum(weights)
    return float(values[np.searchsorted(np.cumsum(weights), cutoff, side="left")])


def build_knn_exclusion_sets(coords, k=100):
    n_cells = coords.shape[0]
    n_neighbors = min(k + 1, n_cells)
    nn = NearestNeighbors(n_neighbors=n_neighbors, algorithm="auto")
    nn.fit(coords)
    neigh = nn.kneighbors(return_distance=False)

    # Exclude self when present. Use Python sets for robust membership checks during sampling.
    exclusion_sets = []
    for i in range(n_cells):
        s = set(neigh[i].tolist())
        s.add(i)
        exclusion_sets.append(s)
    return exclusion_sets


def sample_far_edges_for_true_edges(
    true_edge_index_np,
    true_edge_row_indices,
    celltypes,
    exclusion_sets,
    rng,
    max_attempts=100,
):
    """Sample one far pseudo-edge for each selected true edge.

    Sender is kept fixed. Receiver is sampled from the same receiver cell type but outside
    the sender's KNN-FAR_K exclusion set.
    """
    receiver_indices_by_type = {
        ct: np.where(celltypes == ct)[0].astype(np.int64)
        for ct in np.unique(celltypes)
    }

    far_edges = []
    kept_true_rows = []
    pair_labels = []

    for row_idx in true_edge_row_indices:
        sender = int(true_edge_index_np[row_idx, 0])
        receiver = int(true_edge_index_np[row_idx, 1])
        sender_ct = celltypes[sender]
        receiver_ct = celltypes[receiver]

        pool = receiver_indices_by_type.get(receiver_ct, np.array([], dtype=np.int64))
        if pool.size == 0:
            continue

        exclusion = exclusion_sets[sender]
        chosen = None

        # Fast rejection sampling first.
        for _ in range(max_attempts):
            cand = int(pool[rng.integers(0, pool.size)])
            if cand not in exclusion:
                chosen = cand
                break

        # Exact fallback when rejection sampling fails.
        if chosen is None:
            mask = np.fromiter((int(x) not in exclusion for x in pool), dtype=bool, count=pool.size)
            far_pool = pool[mask]
            if far_pool.size == 0:
                continue
            chosen = int(far_pool[rng.integers(0, far_pool.size)])

        far_edges.append([sender, chosen])
        kept_true_rows.append(int(row_idx))
        pair_labels.append(f"{sender_ct}→{receiver_ct}")

    if len(far_edges) == 0:
        return (
            np.zeros((0, 2), dtype=np.int64),
            np.zeros(0, dtype=np.int64),
            np.asarray([], dtype=object),
        )

    return (
        np.asarray(far_edges, dtype=np.int64),
        np.asarray(kept_true_rows, dtype=np.int64),
        np.asarray(pair_labels, dtype=object),
    )


def compute_top_sender_receiver_pairs_per_mi(resources, original_factors):
    """Identify dominant sender→receiver cell-type pairs for each MI.

    Ranking uses MI_ACTIVITY_COMPARISON_STAT on the original true-neighbor graph.
    With "mean", this is an edge-count-weighted mean across slices.
    With "median", this is an edge-count-weighted median of slice-level medians.
    """
    study_key = resources["study_key"]
    cfg = resources["cfg"]
    adata_list = resources["adata_list"]
    data_list = resources["spidernet_data_list"]
    stat = _get_mi_activity_stat()

    records = []
    for slice_index, (adata, data, factor_orig_raw) in enumerate(zip(adata_list, data_list, original_factors)):
        celltypes = get_celltype_array(adata, cfg["celltype_col"])
        edge_index_np = get_edge_index_np(data)
        pair_labels = _pair_label_from_edges(edge_index_np, celltypes)

        n_mi = factor_orig_raw.shape[1]
        mi_names = mi_column_names(n_mi)

        for pair in np.unique(pair_labels):
            rows = np.where(pair_labels == pair)[0]
            if MIN_TRUE_EDGES_PER_CELLTYPE_PAIR is not None and rows.size < MIN_TRUE_EDGES_PER_CELLTYPE_PAIR:
                continue

            activity_mi = summarize_mi_activity(factor_orig_raw[rows, :], axis=0)
            sender_ct, receiver_ct = pair.split("→", 1)

            for j, mi_name in enumerate(mi_names):
                records.append({
                    "Study": study_key,
                    "SliceIndex": slice_index,
                    "Sender": sender_ct,
                    "Receiver": receiver_ct,
                    "Pair": pair,
                    "MI": mi_name,
                    "n_edges": int(rows.size),
                    "slice_original_true_neighbor_activity": float(activity_mi[j]),
                    "ActivityStat": stat,
                })

    pair_mi_df = pd.DataFrame(records)
    if pair_mi_df.empty:
        return pair_mi_df, pair_mi_df

    if stat == "mean":
        pair_mi_df["sum_original"] = pair_mi_df["slice_original_true_neighbor_activity"] * pair_mi_df["n_edges"]
        ranked = (
            pair_mi_df
            .groupby(["Study", "Sender", "Receiver", "Pair", "MI", "ActivityStat"], as_index=False)
            .agg(
                n_edges_original=("n_edges", "sum"),
                sum_original=("sum_original", "sum"),
            )
        )
        ranked["original_true_neighbor_activity"] = ranked["sum_original"] / ranked["n_edges_original"]
        ranked = ranked.drop(columns=["sum_original"])
    else:
        rows = []
        group_cols = ["Study", "Sender", "Receiver", "Pair", "MI", "ActivityStat"]
        for key, sub in pair_mi_df.groupby(group_cols, sort=False):
            rows.append({
                **dict(zip(group_cols, key)),
                "n_edges_original": int(sub["n_edges"].sum()),
                "original_true_neighbor_activity": weighted_median(
                    sub["slice_original_true_neighbor_activity"].values,
                    sub["n_edges"].values,
                ),
            })
        ranked = pd.DataFrame(rows)

    ranked["_mi_num"] = ranked["MI"].map(_mi_num)
    ranked = ranked.sort_values(["_mi_num", "original_true_neighbor_activity"], ascending=[True, False])

    top_pairs = (
        ranked
        .groupby("MI", group_keys=False)
        .head(TOP_N_PAIRS_PER_MI)
        .copy()
    )
    top_pairs["TopPairRank"] = top_pairs.groupby("MI").cumcount() + 1
    top_pairs = top_pairs.sort_values(["_mi_num", "TopPairRank"]).drop(columns=["_mi_num"])

    return ranked.drop(columns=["_mi_num"]), top_pairs


def select_true_edge_rows_for_target_pairs(
    edge_index_np,
    celltypes,
    target_pairs,
    rng,
    max_edges_per_pair=MAX_TRUE_EDGES_PER_CELLTYPE_PAIR,
):
    pair_labels = _pair_label_from_edges(edge_index_np, celltypes)
    selected_rows = []

    for pair in sorted(set(target_pairs)):
        rows = np.where(pair_labels == pair)[0]
        if rows.size == 0:
            continue
        if max_edges_per_pair is not None and rows.size > max_edges_per_pair:
            rows = rng.choice(rows, size=max_edges_per_pair, replace=False)
        selected_rows.append(rows)

    if len(selected_rows) == 0:
        return np.zeros(0, dtype=np.int64)

    return np.concatenate(selected_rows).astype(np.int64, copy=False)


def run_experiment1_true_vs_far(resources, original_factors):
    study_key = resources["study_key"]
    cfg = resources["cfg"]
    model = resources["model"]
    adata_list = resources["adata_list"]
    data_list = resources["spidernet_data_list"]
    stat = _get_mi_activity_stat()
    stat_label = _activity_stat_label()

    outdir = Path(resources["outdir"]) / "Experiment1_topMIpair_true_neighbor_vs_far_edges"
    outdir.mkdir(parents=True, exist_ok=True)

    # Step 1: identify top sender→receiver cell-type pairs for each MI using original true-neighbor MI.
    all_pair_mi_ranking, top_pair_table = compute_top_sender_receiver_pairs_per_mi(resources, original_factors)

    ranking_csv = outdir / f"{study_key}_Exp1_all_pair_mi_original_{stat}_ranking.csv"
    all_pair_mi_ranking.to_csv(ranking_csv, index=False)
    print("Saved:", ranking_csv)

    top_csv = outdir / f"{study_key}_Exp1_top{TOP_N_PAIRS_PER_MI}_pairs_per_MI_by_original_{stat}.csv"
    top_pair_table.to_csv(top_csv, index=False)
    print("Saved:", top_csv)

    if top_pair_table.empty:
        print("No top sender→receiver cell-type pairs found; skip Experiment 1.")
        return pd.DataFrame(), pd.DataFrame(), top_pair_table

    target_pairs = sorted(top_pair_table["Pair"].unique().tolist())
    target_pair_mi_keys = set(zip(top_pair_table["Pair"], top_pair_table["MI"]))

    rng = np.random.default_rng(RANDOM_SEED)
    records = []
    raw_store = {}

    # Step 2: only sample true/far edges for the union of selected top sender→receiver cell-type pairs.
    for slice_index, (adata, data, factor_orig_raw) in enumerate(zip(adata_list, data_list, original_factors)):
        print(f"Experiment 1 | {study_key} slice {slice_index + 1}/{len(adata_list)}")

        coords = get_spatial_matrix(adata, cfg["spatial_key"])
        celltypes = get_celltype_array(adata, cfg["celltype_col"])
        edge_index_np = get_edge_index_np(data)

        selected_true_rows = select_true_edge_rows_for_target_pairs(
            edge_index_np=edge_index_np,
            celltypes=celltypes,
            target_pairs=target_pairs,
            rng=rng,
            max_edges_per_pair=MAX_TRUE_EDGES_PER_CELLTYPE_PAIR,
        )

        if selected_true_rows.size == 0:
            print("  skipped: none of the selected top pairs are present in this slice.")
            continue

        exclusion_sets = build_knn_exclusion_sets(coords, k=FAR_K)
        far_edges, kept_true_rows, pair_labels = sample_far_edges_for_true_edges(
            true_edge_index_np=edge_index_np,
            true_edge_row_indices=selected_true_rows,
            celltypes=celltypes,
            exclusion_sets=exclusion_sets,
            rng=rng,
        )

        if far_edges.shape[0] == 0:
            print("  skipped: no far pseudo-edges could be sampled.")
            continue

        # Original MI values are raw model outputs inferred on the original graph and then sampled.
        true_mi = factor_orig_raw[kept_true_rows, :]

        # Far pseudo-edge MI values are raw model outputs inferred after replacing edge_index with far edges.
        far_graph = clone_graph_with_new_edges(data, far_edges, zero_old_edge_attrs=True)
        far_mi = run_model_get_mi(model, far_graph, device)

        n_mi = true_mi.shape[1]
        mi_names = mi_column_names(n_mi)

        sample_name = str(adata.obs[cfg["sample_col"]].iloc[0]) if cfg["sample_col"] in adata.obs.columns else f"Slice_{slice_index + 1}"

        for pair in np.unique(pair_labels):
            mask = pair_labels == pair
            n_pair_edges = int(np.sum(mask))
            if n_pair_edges == 0:
                continue

            true_stat = summarize_mi_activity(true_mi[mask, :], axis=0)
            far_stat = summarize_mi_activity(far_mi[mask, :], axis=0)
            log2_enrich = safe_log2_ratio(true_stat, far_stat)

            sender_ct, receiver_ct = pair.split("→", 1)
            for j, mi_name in enumerate(mi_names):
                # Keep records only for the selected MI-specific top pairs.
                if (pair, mi_name) not in target_pair_mi_keys:
                    continue

                key = (study_key, sender_ct, receiver_ct, pair, mi_name)
                if key not in raw_store:
                    raw_store[key] = {"true": [], "far": [], "n_edges": 0}
                raw_store[key]["true"].append(np.asarray(true_mi[mask, j], dtype=float))
                raw_store[key]["far"].append(np.asarray(far_mi[mask, j], dtype=float))
                raw_store[key]["n_edges"] += n_pair_edges

                records.append({
                    "Study": study_key,
                    "SliceIndex": slice_index,
                    "Sample": sample_name,
                    "Sender": sender_ct,
                    "Receiver": receiver_ct,
                    "Pair": pair,
                    "MI": mi_name,
                    "n_edges": n_pair_edges,
                    "ActivityStat": stat,
                    "true_neighbor_activity": float(true_stat[j]),
                    "far_pseudo_activity": float(far_stat[j]),
                    "log2FC_true_vs_far": float(log2_enrich[j]),
                })

    long_df = pd.DataFrame(records)
    if long_df.empty:
        out_csv = outdir / f"{study_key}_Exp1_topMIpair_true_neighbor_vs_far_long_{stat}.csv"
        long_df.to_csv(out_csv, index=False)
        print("Saved:", out_csv)
        return long_df, pd.DataFrame(), top_pair_table

    # Add the original ranking metadata for the selected MI-pair entries.
    meta_cols = [
        "Study", "Sender", "Receiver", "Pair", "MI",
        "TopPairRank", "n_edges_original", "original_true_neighbor_activity", "ActivityStat",
    ]
    long_df = long_df.merge(
        top_pair_table[meta_cols],
        on=["Study", "Sender", "Receiver", "Pair", "MI", "ActivityStat"],
        how="left",
    )

    out_csv = outdir / f"{study_key}_Exp1_topMIpair_true_neighbor_vs_far_long_{stat}.csv"
    long_df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

    # Summary across slices. For mean, this is the edge-level mean across sampled edges.
    # For median, this is the edge-level median across sampled edges.
    summary_rows = []
    for key, val in raw_store.items():
        study, sender, receiver, pair, mi_name = key
        true_values = np.concatenate(val["true"]) if len(val["true"]) > 0 else np.asarray([], dtype=float)
        far_values = np.concatenate(val["far"]) if len(val["far"]) > 0 else np.asarray([], dtype=float)
        true_activity = float(summarize_mi_activity(true_values)) if true_values.size else np.nan
        far_activity = float(summarize_mi_activity(far_values)) if far_values.size else np.nan
        summary_rows.append({
            "Study": study,
            "Sender": sender,
            "Receiver": receiver,
            "Pair": pair,
            "MI": mi_name,
            "n_edges": int(val["n_edges"]),
            "ActivityStat": stat,
            "true_neighbor_activity": true_activity,
            "far_pseudo_activity": far_activity,
            "log2FC_true_vs_far": float(safe_log2_ratio(true_activity, far_activity)),
            "activity_difference_true_minus_far": true_activity - far_activity,
            "far_over_true_ratio": (far_activity + MI_ACTIVITY_EPS) / (true_activity + MI_ACTIVITY_EPS),
        })

    summary = pd.DataFrame(summary_rows)
    summary = summary.merge(
        top_pair_table[meta_cols],
        on=["Study", "Sender", "Receiver", "Pair", "MI", "ActivityStat"],
        how="left",
    )

    summary["_mi_num"] = summary["MI"].map(_mi_num)
    summary = summary.sort_values(["_mi_num", "TopPairRank"]).drop(columns=["_mi_num"])

    out_summary = outdir / f"{study_key}_Exp1_topMIpair_true_neighbor_vs_far_summary_{stat}.csv"
    summary.to_csv(out_summary, index=False)
    print("Saved:", out_summary)
    print(f"Using {stat_label.lower()} MI activity for near/far log2FC.")

    return long_df, summary, top_pair_table


def plot_combined_experiment1_log2fc_barplots(all_summary, outdir=None):
    """Save one PDF with one study per row.

    The panels share the log2FC axis while retaining study-specific MI-pair labels.
    """
    stat = _get_mi_activity_stat()
    stat_label = _activity_stat_label()

    if isinstance(all_summary, list):
        nonempty = [df for df in all_summary if df is not None and not df.empty]
        if len(nonempty) == 0:
            print("No Experiment 1 summaries available; skip combined barplot.")
            return None
        plot_all = pd.concat(nonempty, axis=0, ignore_index=True)
    else:
        plot_all = all_summary.copy()

    if plot_all.empty:
        print("Experiment 1 summary is empty; skip combined barplot.")
        return None

    if outdir is None:
        outdir = Path(COMBINED_OUTPUT_DIR)
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    set_plot_style()

    # Keep study order consistent with user controls when possible.
    study_order = [s for s in STUDIES_TO_RUN if s in set(plot_all["Study"].astype(str))]
    study_order += [s for s in plot_all["Study"].astype(str).unique().tolist() if s not in study_order]

    n_studies = len(study_order)
    if n_studies == 0:
        print("No studies found in Experiment 1 summary; skip combined barplot.")
        return None

    max_rows = 1
    prepared = {}
    for study in study_order:
        sub = plot_all.loc[plot_all["Study"].astype(str) == str(study)].copy()
        if sub.empty:
            continue
        sub["_mi_num"] = sub["MI"].map(_mi_num)
        sub = sub.sort_values(["_mi_num", "TopPairRank"]).reset_index(drop=True)
        sub["ComparisonLabel"] = (
            sub["MI"].astype(str)
            + " | "
            + sub["Pair"].map(lambda x: _shorten_label(x, max_chars=MAX_LABEL_CHARS))
        )
        prepared[study] = sub
        max_rows = max(max_rows, sub.shape[0])

    if len(prepared) == 0:
        print("No non-empty study summaries found; skip combined barplot.")
        return None

    row_heights = [max(1, prepared[study].shape[0]) for study in prepared]
    fig_w = COMBINED_BARPLOT_WIDTH
    fig_h = max(6.0, 0.16 * sum(row_heights) + 2.1)
    fig, axes = plt.subplots(
        len(prepared),
        1,
        figsize=(fig_w, fig_h),
        sharex=True,
        squeeze=False,
        gridspec_kw={"height_ratios": row_heights},
    )
    axes = axes[:, 0]

    all_vals = plot_all["log2FC_true_vs_far"].to_numpy(float)
    finite_vals = all_vals[np.isfinite(all_vals)]
    if finite_vals.size > 0:
        xmax = float(np.nanpercentile(np.abs(finite_vals), 98))
        xmax = max(xmax, 0.15)
    else:
        xmax = 0.15
    xmax *= 1.10

    for ax, study in zip(axes, prepared.keys()):
        sub = prepared[study]
        y_positions = np.arange(sub.shape[0])[::-1]
        vals = sub["log2FC_true_vs_far"].to_numpy(float)
        colors = np.where(vals >= 0, "#d62728", "#1f77b4")

        ax.barh(y_positions, vals, color=colors, edgecolor="none", alpha=0.85)
        ax.axvline(0, color="black", linewidth=0.7)
        ax.set_yticks(y_positions)
        ax.set_yticklabels(sub["ComparisonLabel"].tolist(), fontsize=4.5)
        ax.set_xlim(-xmax, xmax)
        ax.set_xlabel(f"log2FC\n{stat} near / far", fontsize=7)
        ax.set_title(str(study), fontsize=8, fontweight="bold")
        ax.grid(axis="x", linewidth=0.3, alpha=0.4)
        sns.despine(ax=ax, left=False, bottom=False)

    fig.suptitle(
        f"True-neighbor enrichment of dominant MI-specific cell-type pairs ({stat_label.lower()} MI activity)",
        fontsize=9,
        y=0.995,
    )
    plt.tight_layout(h_pad=1.2, rect=[0, 0, 1, 0.97])

    out_pdf = outdir / (
        f"Combined_Exp1_top{TOP_N_PAIRS_PER_MI}_pair_per_MI_"
        f"log2FC_{stat}_barplots_two_study_rows.pdf"
    )
    fig.savefig(out_pdf, bbox_inches="tight")
    print("Saved:", out_pdf)
    plt.show()
    return out_pdf

## 6. Run or reload the spatial control, then save the combined plot

In the default mode, load each study once, infer activities on its original graph once, and evaluate the matched distant edges. The per-study CSVs are written below the run directory recorded in `run_dirs.json`, inside `Spatial_false_positive_null_benchmark/Experiment1_topMIpair_true_neighbor_vs_far_edges/`.

With `PLOT_ONLY=True`, read each existing `*_Exp1_topMIpair_true_neighbor_vs_far_summary_mean.csv` (or the selected statistic) and skip checkpoint loading and inference. These summaries must come from the intended run and the same analysis settings. Both modes use the same combined summary, panel report, and plotting code below.

The combined PDF, summary CSV and panel-report CSV are saved under the local
`output/executed/` directory, alongside the execution notebook saved by the runner.


In [ ]:
all_exp1_long = []
all_exp1_summary = []
all_exp1_top_pairs = []

for study_key in STUDIES_TO_RUN:
    if PLOT_ONLY:
        run_dirs = load_run_dirs(STUDY_CONFIG[study_key]["output_root"])
        summary_path = (
            Path(run_dirs["run_dir"])
            / "Spatial_false_positive_null_benchmark"
            / "Experiment1_topMIpair_true_neighbor_vs_far_edges"
            / f"{study_key}_Exp1_topMIpair_true_neighbor_vs_far_summary_{_get_mi_activity_stat()}.csv"
        )
        if not summary_path.exists():
            raise FileNotFoundError(
                f"Missing spatial summary: {summary_path}. Run with PLOT_ONLY=False first."
            )
        exp1_summary = pd.read_csv(summary_path, float_precision="round_trip")
        if not exp1_summary.empty:
            if set(exp1_summary["Study"].astype(str)) != {study_key}:
                raise ValueError(f"Unexpected study in spatial summary: {summary_path}")
            if set(exp1_summary["ActivityStat"].astype(str)) != {_get_mi_activity_stat()}:
                raise ValueError(f"Activity statistic does not match controls: {summary_path}")
        all_exp1_summary.append(exp1_summary)
        print("Loaded spatial summary:", summary_path)
    else:
        resources = load_study_resources(study_key, STUDY_CONFIG[study_key])

        # Cache original raw MI inference once per study.
        original_factors = infer_original_mi_per_slice(resources)

        exp1_long, exp1_summary, exp1_top_pairs = run_experiment1_true_vs_far(
            resources=resources,
            original_factors=original_factors,
        )

        all_exp1_long.append(exp1_long)
        all_exp1_summary.append(exp1_summary)
        all_exp1_top_pairs.append(exp1_top_pairs)

# Save combined tables and the only final figure: one study per row.
combined_outdir = Path(COMBINED_OUTPUT_DIR)
combined_outdir.mkdir(parents=True, exist_ok=True)

nonempty_summary = [df for df in all_exp1_summary if df is not None and not df.empty]
if len(nonempty_summary) > 0:
    combined_exp1_summary = pd.concat(nonempty_summary, axis=0, ignore_index=True)
    combined_summary_path = combined_outdir / f"Combined_Exp1_top{TOP_N_PAIRS_PER_MI}_true_neighbor_vs_far_summary_{_get_mi_activity_stat()}.csv"
    combined_exp1_summary.to_csv(combined_summary_path, index=False)
    print("Saved:", combined_summary_path)

    # Report positive-log2FC MI counts and median log2FC for each study panel.
    panel_report_rows = []
    panel_order = [
        study for study in STUDIES_TO_RUN
        if study in combined_exp1_summary["Study"].astype(str).unique()
    ]
    for study in panel_order:
        panel_df = combined_exp1_summary[
            combined_exp1_summary["Study"].astype(str) == str(study)
        ].copy()
        panel_df["log2FC_true_vs_far"] = pd.to_numeric(
            panel_df["log2FC_true_vs_far"], errors="coerce"
        )
        finite_panel_df = panel_df.dropna(subset=["log2FC_true_vs_far"])
        positive_panel_df = finite_panel_df[
            finite_panel_df["log2FC_true_vs_far"] > 0
        ]
        panel_report_rows.append({
            "Study": study,
            "n_total_MI": int(finite_panel_df["MI"].nunique()),
            "n_positive_log2FC_MI": int(positive_panel_df["MI"].nunique()),
            "median_log2FC": float(finite_panel_df["log2FC_true_vs_far"].median())
            if not finite_panel_df.empty else np.nan,
        })

    combined_exp1_panel_report = pd.DataFrame(panel_report_rows)
    print("\nExperiment 1 panel report: positive-log2FC MI count and median log2FC")
    display(combined_exp1_panel_report)
    combined_panel_report_path = combined_outdir / (
        f"Combined_Exp1_top{TOP_N_PAIRS_PER_MI}_panel_positive_MI_and_median_log2FC_"
        f"{_get_mi_activity_stat()}.csv"
    )
    combined_exp1_panel_report.to_csv(combined_panel_report_path, index=False)
    print("Saved:", combined_panel_report_path)

    plot_combined_experiment1_log2fc_barplots(combined_exp1_summary, combined_outdir)
else:
    print("No non-empty Experiment 1 summary tables found.")


In [ ]:
# ==============================================================
# Summary: positive / negative log2FC proportions per study
# --------------------------------------------------------------
# Based on all_exp1_summary.
# Positive log2FC means true neighboring edges have higher MI
# activity than far pseudo-edges.
# ==============================================================

# all_exp1_summary may be either a list of dataframes or one dataframe
if isinstance(all_exp1_summary, list):
    exp1_summary_df = pd.concat(all_exp1_summary, axis=0, ignore_index=True)
else:
    exp1_summary_df = all_exp1_summary.copy()

# Identify columns robustly
logfc_candidates = [
    "log2FC_true_vs_far",
    "log2FC_true_neighbor_vs_far",
    "log2FC",
]
logfc_col = next((c for c in logfc_candidates if c in exp1_summary_df.columns), None)
if logfc_col is None:
    raise ValueError(
        f"Cannot find log2FC column. Available columns: {list(exp1_summary_df.columns)}"
    )

study_candidates = [
    "study_key",
    "study",
    "Study",
    "dataset",
    "Dataset",
]
study_col = next((c for c in study_candidates if c in exp1_summary_df.columns), None)
if study_col is None:
    raise ValueError(
        f"Cannot find study column. Available columns: {list(exp1_summary_df.columns)}"
    )

# Keep finite log2FC values only
df = exp1_summary_df[[study_col, logfc_col]].copy()
df = df[np.isfinite(df[logfc_col].to_numpy())].copy()

eps_zero = 1e-12

print("Positive / negative log2FC proportions by study")
print("Positive: true-neighbor MI > far-pseudo-edge MI")
print("Negative: true-neighbor MI < far-pseudo-edge MI")
print("=" * 80)

summary_rows = []

for study, sub in df.groupby(study_col):
    vals = sub[logfc_col].to_numpy()
    n_total = len(vals)

    n_pos = int(np.sum(vals > eps_zero))
    n_neg = int(np.sum(vals < -eps_zero))
    n_zero = int(np.sum(np.abs(vals) <= eps_zero))

    pos_prop = n_pos / n_total if n_total > 0 else np.nan
    neg_prop = n_neg / n_total if n_total > 0 else np.nan
    zero_prop = n_zero / n_total if n_total > 0 else np.nan

    mean_logfc = float(np.mean(vals)) if n_total > 0 else np.nan
    median_logfc = float(np.median(vals)) if n_total > 0 else np.nan

    print(f"\nStudy: {study}")
    print(f"  Total MI-pair tests: {n_total}")
    print(f"  Positive log2FC: {n_pos} / {n_total} ({pos_prop:.2%})")
    print(f"  Negative log2FC: {n_neg} / {n_total} ({neg_prop:.2%})")
    print(f"  Zero log2FC:     {n_zero} / {n_total} ({zero_prop:.2%})")
    print(f"  Mean log2FC:     {mean_logfc:.4f}")
    print(f"  Median log2FC:   {median_logfc:.4f}")

    summary_rows.append({
        "study": study,
        "n_total": n_total,
        "n_positive": n_pos,
        "n_negative": n_neg,
        "n_zero": n_zero,
        "positive_prop": pos_prop,
        "negative_prop": neg_prop,
        "zero_prop": zero_prop,
        "mean_log2FC": mean_logfc,
        "median_log2FC": median_logfc,
    })

exp1_logfc_direction_summary = pd.DataFrame(summary_rows)
display(exp1_logfc_direction_summary)